# Fabric IQ and Foundry IQ agents — companion notebook

Companion to the end-to-end deck **Microsoft Foundry Fabric IQ and Foundry IQ Agents** 
One scenario runs through every step: the operations centre of the fictional **Valmont Winter Games**, built from the kit sample `samples/games-operations` (venues, sessions, assets, live telemetry, inspection findings and 51 PDF documents).

This first part covers the knowledge layer: the two Fabric MCP endpoints, a document indexing pipeline in Azure AI
Search, and Foundry IQ knowledge sources side by side. Steps 5 to 11 (agents, evaluations, permissions, Work IQ,
publishing, Copilot Studio, API Management) follow in later updates of this notebook.

**Before you start (Fabric side)** — built with the sample, in this order
- `GamesLakehouse` loaded by `samples/games-operations/fabric/01_load_lakehouse.ipynb`, with the PDFs under
  **Files** → `games_ops/output/documents` (four subfolders: manuals, inspections, transport, policies)
- `GamesEventhouse` (`fabric/02_eventhouse_setup.kql`), the semantic model **Games Operations Model**
  (`fabric/03_build_semantic_model.ipynb`), the findings table (`fabric/04_extract_findings.ipynb`) and the
  ontology `GamesOperationsOntology` finished with the three prompts in `ontology/ontology-agent-prompts.md`
- A Fabric capacity F2 or higher; the workspace is not *My workspace*; the lakehouse has no sensitivity label
  (as per official documentation, the OneLake indexer can't read labelled Fabric items)

**Before you start (Azure side)**
- An Azure AI Search service, **Basic tier or higher**, semantic ranker enabled, with a **system-assigned managed
  identity**, in the same Microsoft Entra tenant as the Fabric workspace
- The Search service identity has **Contributor** on the Fabric workspace (workspace **Manage access**)
- The Search service identity has **Cognitive Services User** and **Cognitive Services OpenAI User** on the Foundry
  resource (content extraction billing, embeddings, knowledge base model)
- The Foundry resource is in a region that supports Azure Content Understanding (used by step 3)
- A Foundry project with a chat model deployment (`FOUNDRY_MODEL_NAME`, a model supported by knowledge bases, for
  example `gpt-5-mini` or `gpt-4.1-mini`) and an embedding deployment (`FOUNDRY_EMBEDDING_NAME`,
  `text-embedding-3-small` or `text-embedding-3-large`)
- Your own account has **Search Service Contributor** and **Search Index Data Contributor** on the search service,
  read access to the Fabric workspace and ontology, and is signed in with `az login` (the ontology steps use your
  sign-in; as per official documentation, service principals can't call the ontology MCP endpoint or use Fabric
  IQ MCP)
- The generated sample output in `samples/games-operations/output` (step 0 regenerates the register and questions
  if it is missing)

| Step | What it shows | Status | Deck section |
|---|---|---|---|
| 0 | Settings, clients and the sample's evaluation questions | GA | — |
| 1 | The ontology as an MCP server: its tools and one question | preview | 2 |
| 2 | Fabric IQ MCP over the semantic model: its tools and one discovery call | GA | 2 |
| 3 | Document indexing pipeline: OneLake data source, skillset, index, indexer, chunk inspection | GA | 4 |
| 4 | Knowledge sources side by side: custom index, indexed OneLake, Fabric ontology; one knowledge base | preview | 3 |
| — | Cleanup of the resources created in steps 3 and 4 | GA | — |

Chargeable cells start with `# CHARGEABLE:` and destructive cells with `# DESTRUCTIVE:`. Azure calls run only when
every required setting is present (`LIVE`).

Lines marked `# verify` use a shape that was documented at the time of writing but not executed here — check them against the Learn page linked in the cell before you present.

## 0 · Settings and clients

One `.env` at the repository root, shared with the other kit notebooks. This notebook adds
`AZURE_SEARCH_ENDPOINT` and the Fabric item IDs (`FABRIC_*`); the IDs are in the item URLs in the Fabric portal,
for example `https://app.fabric.microsoft.com/groups/<workspace-id>/ontologies/<ontology-id>`.
Authentication is Microsoft Entra ID through `DefaultAzureCredential`; no keys. Install the shared
dependencies once from the repository root with `python3 -m pip install -r requirements.txt` (see Local Setup in
the kit README); it includes the sample's requirements.

Run with the repository root as the working directory. The first cell only loads settings and prints the exact
`.env` path and any missing names; it makes no Azure calls. Save `.env` and rerun that cell after edits; no kernel
restart is needed. File values override existing environment variables. A missing `FABRIC_WORKSPACE_ID` must be
added to `.env` explicitly: the lakehouse and ontology IDs do not supply the workspace ID.
Each Fabric ID must be just a GUID. For `FABRIC_ONTOLOGY_ID`, copy only the GUID after `/ontologies/`,
not any following `/entity/<id>` path or URL query parameters.

In [ ]:
import os
from pathlib import Path
from uuid import UUID
from dotenv import load_dotenv

ENV_FILE = Path(".env").resolve()
if not ENV_FILE.is_file():
    raise FileNotFoundError(f"No .env at {ENV_FILE}. Run from the repository root and save your .env there.")
load_dotenv(ENV_FILE, override=True)
print(f"Settings file: {ENV_FILE}")

def env(name, default=""):
    return (os.getenv(name) or default).strip()

FOUNDRY_PROJECT_ENDPOINT = env("FOUNDRY_PROJECT_ENDPOINT")
FOUNDRY_MODEL_NAME = env("FOUNDRY_MODEL_NAME")
FOUNDRY_EMBEDDING_NAME = env("FOUNDRY_EMBEDDING_NAME")
AZURE_SEARCH_ENDPOINT = env("AZURE_SEARCH_ENDPOINT")
FABRIC_WORKSPACE_ID = env("FABRIC_WORKSPACE_ID")
FABRIC_LAKEHOUSE_ID = env("FABRIC_LAKEHOUSE_ID")
FABRIC_ONTOLOGY_ID = env("FABRIC_ONTOLOGY_ID")
FABRIC_DOCS_FOLDER = env("FABRIC_DOCS_FOLDER", "games_ops/output/documents")

required = ("FOUNDRY_PROJECT_ENDPOINT", "FOUNDRY_MODEL_NAME", "FOUNDRY_EMBEDDING_NAME",
            "AZURE_SEARCH_ENDPOINT", "FABRIC_WORKSPACE_ID", "FABRIC_LAKEHOUSE_ID", "FABRIC_ONTOLOGY_ID")
missing = [name for name in required if not env(name) or "<" in env(name)]
invalid_ids = []
for name in ("FABRIC_WORKSPACE_ID", "FABRIC_LAKEHOUSE_ID", "FABRIC_ONTOLOGY_ID"):
    if name not in missing:
        try:
            UUID(env(name))
        except ValueError:
            invalid_ids.append(name)
LIVE = not missing and not invalid_ids
print("Required settings missing:", ", ".join(missing) if missing else "none")
if invalid_ids:
    print("Invalid Fabric IDs:", ", ".join(invalid_ids), "— use GUIDs only, without /entity/... or URL parameters.")

### Clients and model metadata

Run this cell after checking the settings above. Model lookups use your Azure sign-in and only run when
`LIVE` is true. Keep model names separate from deployment names: Azure AI Search needs both.

In [ ]:
import sys, json, time, re, subprocess
from urllib.parse import urlparse
import pandas as pd
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

credential = DefaultAzureCredential()            # your az login identity; the Fabric steps need a user, not a service principal
ACCOUNT = (urlparse(FOUNDRY_PROJECT_ENDPOINT).hostname or "account").split(".")[0]
FOUNDRY_OPENAI_URL = f"https://{ACCOUNT}.openai.azure.com"          # embeddings, vectorizer and knowledge base model
FOUNDRY_SERVICES_URL = f"https://{ACCOUNT}.services.ai.azure.com"   # resource that bills content extraction in step 3

CHAT_MODEL, EMBED_MODEL = FOUNDRY_MODEL_NAME, FOUNDRY_EMBEDDING_NAME
if LIVE:
    project = AIProjectClient(endpoint=FOUNDRY_PROJECT_ENDPOINT, credential=credential)
    for var in ("CHAT_MODEL", "EMBED_MODEL"):
        deployment = globals()[var]
        try:   # the search service needs the model name as well as the deployment name
            globals()[var] = project.deployments.get(deployment).model_name   # verify https://learn.microsoft.com/en-us/python/api/azure-ai-projects/
        except Exception as e:
            print(f"Model name for {deployment} not read ({type(e).__name__}); using the deployment name")
EMBED_DIMS = {"text-embedding-3-large": 3072, "text-embedding-3-small": 1536, "text-embedding-ada-002": 1536}.get(EMBED_MODEL, 1536)
print(f"chat model {CHAT_MODEL} · embedding model {EMBED_MODEL} ({EMBED_DIMS} dimensions)")

The sample's document register and evaluation questions come from the generated output. The register adds the
document type, venue, version and status to every chunk in step 3; the questions and expected answers drive the
comparisons in step 4 and the evaluations in step 6. Generation is deterministic, so the IDs match the data
loaded into Fabric.

In [ ]:
SAMPLE = Path("samples/games-operations")
SAMPLE_OUT = SAMPLE / "output"
if not (SAMPLE_OUT / "lakehouse" / "document_register.csv").exists():
    print("Sample output not found; generating the register and questions (no PDFs) ...")
    subprocess.run([sys.executable, str(SAMPLE / "generator" / "generate.py"), "--out", str(SAMPLE_OUT), "--no-docs"], check=True)

register = pd.read_csv(SAMPLE_OUT / "lakehouse" / "document_register.csv", dtype=str, keep_default_na=False)
questions = pd.read_csv(SAMPLE_OUT / "eval" / "questions.csv", dtype=str, keep_default_na=False).set_index("id")
print(f"{len(register)} documents in the register, {len(questions)} evaluation questions")
display(register.groupby(["document_type", "document_status"]).size().rename("documents").reset_index())

# Names of everything this notebook creates in Azure AI Search
INDEX, DATASOURCE, SKILLSET, INDEXER = "games-docs", "games-docs-onelake", "games-docs-skillset", "games-docs-indexer"
KS_DOCS, KS_ONELAKE, KS_ONTOLOGY, KB = "games-docs-ks", "games-onelake-ks", "games-ontology-ks", "games-kb"
SEARCH_API_VERSION = "2026-08-01-preview"   # knowledge bases with answer synthesis, reasoning effort and the Fabric ontology source are preview

## 1 · Ontology as an MCP server (preview)  *(deck section 2)*

Fabric exposes each ontology item as a Model Context Protocol (MCP) endpoint. An agent connected to it can list
the entity types and ask questions in business terms; the ontology resolves them against its bindings
(lakehouse, eventhouse, semantic model). The call runs with your own sign-in: as per official documentation, a
service principal can't call the ontology MCP endpoint yet, and the capacity must be F2 or higher.

Source: https://learn.microsoft.com/en-us/fabric/iq/ontology/how-to-use-ontology-mcp-server

Use the official Python MCP SDK directly: 
- `ClientSession.list_tools()` discovers the server's tools and
- `ClientSession.call_tool()` invokes them. 

### Connect and discover the ontology's tools

Check the configured item before opening the MCP session.

In [ ]:
import httpx
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

FABRIC_SCOPE = "https://api.fabric.microsoft.com/.default"
ONTOLOGY_MCP_URL = (f"https://api.fabric.microsoft.com/v1/mcp/dataPlane/workspaces/{FABRIC_WORKSPACE_ID}"
                    f"/items/{FABRIC_ONTOLOGY_ID}/ontologyEndpoint")
ontology_tools = []
if LIVE:
    headers = {"Authorization": f"Bearer {credential.get_token(FABRIC_SCOPE).token}",
               "x-ms-fabric-skill": "fabriciq-ontology-cli"}
    item_url = f"https://api.fabric.microsoft.com/v1/workspaces/{FABRIC_WORKSPACE_ID}/items/{FABRIC_ONTOLOGY_ID}"
    async with httpx.AsyncClient(headers=headers, timeout=30) as client:
        response = await client.get(item_url)
        if response.status_code == 404:
            raise ValueError("Ontology item not found. Check FABRIC_WORKSPACE_ID and FABRIC_ONTOLOGY_ID in .env.")
        response.raise_for_status()
        item = response.json()
    if item["type"] != "Ontology":
        raise ValueError(f"FABRIC_ONTOLOGY_ID points to a {item['type']}, not an Ontology.")
    print(f"Ontology: {item['displayName']}")
    async with streamablehttp_client(ONTOLOGY_MCP_URL, headers=headers) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            ontology_tools = (await session.list_tools()).tools
    display(pd.DataFrame([t.model_dump(include={"name", "description", "inputSchema"}) for t in ontology_tools]))
else:
    print("Settings missing — step 1 skipped")

One question through the ontology: evaluation question Q12 depends on the `ServiceOverdue` rule, so the answer shows whether the rule travels with the ontology. Call the server's `ask_ontology` tool with its published `request` 

In [ ]:
if LIVE:
    if "ask_ontology" not in {tool.name for tool in ontology_tools}:
        raise RuntimeError("ask_ontology was not discovered. Run the ontology connection cell first.")
    q = questions.loc["Q12"]
    print(f"Tool ask_ontology · question: {q.question}\n")
    headers = {"Authorization": f"Bearer {credential.get_token(FABRIC_SCOPE).token}",
               "x-ms-fabric-skill": "fabriciq-ontology-cli"}
    async with streamablehttp_client(ONTOLOGY_MCP_URL, headers=headers) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool("ask_ontology", {"request": q.question})
    answer = "\n".join(content.text for content in result.content if content.type == "text")
    if result.isError:
        raise RuntimeError(f"ask_ontology failed: {answer}")
    print(answer[:3000])
    print("\nExpected:", q.expected_answer)

## 2 · Fabric IQ MCP over the semantic model  *(deck section 2)*

Fabric IQ MCP is the generally available, tenant-wide MCP server for Power BI semantic models: an agent discovers
models and reports, reads a model's schema and runs DAX queries, with the user's permissions (row-level security
applies). As per official documentation it doesn't reach ontologies or data agents yet, so it complements step 1
rather than replacing it. The `X-Variants` header pins the tool contract to version 1.

| Endpoint | What an agent reaches | Status |
|---|---|---|
| Ontology MCP (step 1) | One ontology: entity types, relationships, rules, live readings | Preview |
| Fabric IQ MCP (this step) | Every semantic model and report the user can read; DAX queries | GA |

Source: https://learn.microsoft.com/en-us/fabric/iq/connectors/fabric-iq-mcp

This cell uses the same SDK directly and can run after step 0 without running step 1. It discovers the
published tools, then calls `DiscoverArtifacts` with its `searchQuery` argument in the same MCP session.

In [ ]:
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

POWERBI_SCOPE = "https://analysis.windows.net/powerbi/api/.default"
FABRIC_IQ_MCP_URL = "https://fabriciq.svc.cloud.microsoft/v1/mcp/fabriciq"   # private link tenants: https://api.fabric.microsoft.com/v1/mcp/fabriciq
fabric_iq_tools = []
if LIVE:
    headers = {"Authorization": f"Bearer {credential.get_token(POWERBI_SCOPE).token}",
               "X-Variants": "Fabric.Routing.FabricIQ.V1",
               "x-ms-fabric-skill": "fabriciq-ontology-cli"}
    async with streamablehttp_client(FABRIC_IQ_MCP_URL, headers=headers) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            fabric_iq_tools = (await session.list_tools()).tools
            display(pd.DataFrame([t.model_dump(include={"name", "description", "inputSchema"}) for t in fabric_iq_tools]))
            if "DiscoverArtifacts" not in {tool.name for tool in fabric_iq_tools}:
                raise RuntimeError("DiscoverArtifacts was not exposed by the Fabric IQ MCP server.")
            result = await session.call_tool("DiscoverArtifacts", {"searchQuery": "Games Operations Model"})
    discovery = "\n".join(content.text for content in result.content if content.type == "text")
    if result.isError:
        raise RuntimeError(f"DiscoverArtifacts failed: {discovery}")
    print(discovery[:2000])
else:
    print("Settings missing — step 2 skipped")

## 3 · Document indexing pipeline  *(deck section 4)*

The ontology answers *which, how many, related to what*. The documents answer *what exactly does the manual or
the inspector say*. This step indexes the 51 PDFs straight from the lakehouse **Files** area, with no copy:

| Object | Role in the pipeline |
|---|---|
| Data source (`onelake`) | Points at the workspace, the lakehouse and the documents folder |
| Skillset | Extracts text and layout, cuts it into chunks, adds an embedding to each chunk |
| Index projections | Write one search document per chunk, linked to its parent file |
| Index | Text, vector and filter fields; semantic configuration and vectorizer for agentic retrieval |
| Indexer | Runs the pipeline and tracks changed files on later runs |

Extraction uses the **Azure Content Understanding skill**, which as per official documentation replaces the
Document Layout skill for new skillsets: it returns Markdown chunks with page numbers, so no separate split
skill is needed. It reads the file bytes (`file_data`); verify that your OneLake indexer supplies them. The
alternative `CHUNKER = "builtIn"` uses the indexer's PDF text extraction plus Text Split (no page citations).
Switching chunkers is explicit: set `RESET_INDEXER = True` and rerun configuration, run, status and enrichment.
Reset reprocesses all PDFs and bills extraction/embeddings again; there is no automatic second run.

Use the Azure AI Search SDK models and client methods directly: define the index and skills, start the indexer,
then inspect its status. The configuration cells make no service calls; only the run cell starts billable work.

Sources: https://learn.microsoft.com/en-us/azure/search/search-how-to-index-onelake-files ·
https://learn.microsoft.com/en-us/azure/search/cognitive-search-skill-content-understanding ·
https://learn.microsoft.com/en-us/azure/search/search-how-to-semantic-chunking-content-understanding ·
https://learn.microsoft.com/en-us/azure/search/agentic-retrieval-how-to-create-index

In [ ]:
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient, SearchIndexerClient, models

if LIVE:
    index_client = SearchIndexClient(endpoint=AZURE_SEARCH_ENDPOINT, credential=credential, api_version=SEARCH_API_VERSION)
    indexer_client = SearchIndexerClient(endpoint=AZURE_SEARCH_ENDPOINT, credential=credential, api_version=SEARCH_API_VERSION)
    search_client = SearchClient(endpoint=AZURE_SEARCH_ENDPOINT, index_name=INDEX, credential=credential, api_version=SEARCH_API_VERSION)

index_definition = models.SearchIndex(
    name=INDEX,
    fields=[
        models.SearchableField(name="chunk_id", key=True, filterable=True, sortable=True, analyzer_name="keyword"),
        models.SearchableField(name="chunk"),
        models.SearchField(name="text_vector", type="Collection(Edm.Single)", searchable=True, hidden=True,
                           stored=False, vector_search_dimensions=EMBED_DIMS, vector_search_profile_name="games-hnsw-profile"),
        models.SearchableField(name="file_name", filterable=True, facetable=True),
        models.SearchableField(name="title", filterable=True, facetable=True),
        # Match the original REST defaults; these attributes cannot change on an existing field.
        models.SimpleField(name="page_from", type="Edm.Int32", filterable=True, sortable=True, facetable=True),
        models.SimpleField(name="page_to", type="Edm.Int32", filterable=True, sortable=True, facetable=True),
        *[models.SimpleField(name=name, type="Edm.String", filterable=True, facetable=True)
          for name in ("parent_id", "document_id", "document_type", "venue_id", "version", "document_status", "sensitivity")],
    ],
    vector_search=models.VectorSearch(
        algorithms=[models.HnswAlgorithmConfiguration(name="games-hnsw")],
        profiles=[models.VectorSearchProfile(name="games-hnsw-profile", algorithm_configuration_name="games-hnsw",
                                             vectorizer_name="games-vectorizer")],
        vectorizers=[models.AzureOpenAIVectorizer(vectorizer_name="games-vectorizer",
            parameters=models.AzureOpenAIVectorizerParameters(resource_url=FOUNDRY_OPENAI_URL,
                deployment_name=FOUNDRY_EMBEDDING_NAME, model_name=EMBED_MODEL))]),
    semantic_search=models.SemanticSearch(default_configuration_name="games-semantic", configurations=[
        models.SemanticConfiguration(name="games-semantic", prioritized_fields=models.SemanticPrioritizedFields(
            title_field=models.SemanticField(field_name="title"),
            content_fields=[models.SemanticField(field_name="chunk")],
            keywords_fields=[models.SemanticField(field_name="document_type"), models.SemanticField(field_name="venue_id")]))]),
)

### Configure the data source, skills and indexer

The search service reads OneLake with its own managed identity (Contributor on
the workspace). Content extraction is billed to the Foundry resource attached to the skillset by identity, with no
key; as per official documentation, the Content Understanding skill has no free daily allowance.

In [ ]:
CHUNKER = "contentUnderstanding"   # or "builtIn": PDF text extraction plus Text Split
RESET_INDEXER = False             # True when switching CHUNKER: reprocess all PDFs (billable)

datasource = models.SearchIndexerDataSourceConnection(
    name=DATASOURCE, type="onelake", connection_string=f"ResourceId={FABRIC_WORKSPACE_ID}",
    container=models.SearchIndexerDataContainer(name=FABRIC_LAKEHOUSE_ID, query=FABRIC_DOCS_FOLDER))

if CHUNKER == "contentUnderstanding":
    context = "/document/text_sections/*"
    text_source = f"{context}/content"
    chunking_skill = models.ContentUnderstandingSkill(
        name="extract-and-chunk", context="/document", extraction_options=["locationMetadata"],
        chunking_properties=models.ContentUnderstandingSkillChunkingProperties(unit="characters", maximum_length=2000, overlap_length=200),
        inputs=[models.InputFieldMappingEntry(name="file_data", source="/document/file_data")],
        outputs=[models.OutputFieldMappingEntry(name="text_sections", target_name="text_sections")])
    page_mappings = [models.InputFieldMappingEntry(name="page_from", source=f"{context}/locationMetadata/pageNumberFrom"),
                     models.InputFieldMappingEntry(name="page_to", source=f"{context}/locationMetadata/pageNumberTo")]
elif CHUNKER == "builtIn":
    context = text_source = "/document/pages/*"
    chunking_skill = models.SplitSkill(
        name="chunk", context="/document", text_split_mode="pages", maximum_page_length=2000,
        page_overlap_length=200, default_language_code="en",
        inputs=[models.InputFieldMappingEntry(name="text", source="/document/content")],
        outputs=[models.OutputFieldMappingEntry(name="textItems", target_name="pages")])
    page_mappings = []
else:
    raise ValueError("CHUNKER must be 'contentUnderstanding' or 'builtIn'.")

embedding_skill = models.AzureOpenAIEmbeddingSkill(
    name="embed", context=context, resource_url=FOUNDRY_OPENAI_URL,
    deployment_name=FOUNDRY_EMBEDDING_NAME, model_name=EMBED_MODEL, dimensions=EMBED_DIMS,
    inputs=[models.InputFieldMappingEntry(name="text", source=text_source)],
    outputs=[models.OutputFieldMappingEntry(name="embedding", target_name="text_vector")])
skillset = models.SearchIndexerSkillset(
    name=SKILLSET, skills=[chunking_skill, embedding_skill],
    cognitive_services_account=models.AIServicesAccountIdentity(subdomain_url=FOUNDRY_SERVICES_URL),
    index_projection=models.SearchIndexerIndexProjection(selectors=[models.SearchIndexerIndexProjectionSelector(
        target_index_name=INDEX, parent_key_field_name="parent_id", source_context=context,
        mappings=[models.InputFieldMappingEntry(name="chunk", source=text_source), *page_mappings,
                  models.InputFieldMappingEntry(name="text_vector", source=f"{context}/text_vector"),
                  models.InputFieldMappingEntry(name="file_name", source="/document/metadata_storage_name")])],
        parameters=models.SearchIndexerIndexProjectionsParameters(projection_mode="skipIndexingParentDocuments")))
indexer = models.SearchIndexer(
    name=INDEXER, data_source_name=DATASOURCE, target_index_name=INDEX, skillset_name=SKILLSET,
    parameters=models.IndexingParameters(configuration=models.IndexingParametersConfiguration(
        data_to_extract="contentAndMetadata", parsing_mode="default", indexed_file_name_extensions=".pdf",
        allow_skillset_to_read_file_data=CHUNKER == "contentUnderstanding")),
    field_mappings=[], output_field_mappings=[])

### Create or update the pipeline and run once

A new indexer starts automatically. An existing indexer needs an explicit run; reset it only when you want to
reprocess unchanged PDFs (for example after switching chunkers). Run this cell once, then use the status cell.

In [ ]:
# CHARGEABLE: content extraction per page and embeddings per chunk (51 short PDFs).
from azure.core.exceptions import ResourceNotFoundError

if LIVE:
    try:
        status = indexer_client.get_indexer_status(INDEXER)
    except ResourceNotFoundError:
        status = None
    if status and status.last_result and status.last_result.status == "inProgress":
        raise RuntimeError("The indexer is already running. Wait for it to finish before changing the pipeline.")
    previous_end = status.last_result.end_time if status and status.last_result else None
    index_client.create_or_update_index(index_definition)
    indexer_client.create_or_update_data_source_connection(datasource)
    indexer_client.create_or_update_skillset(skillset)
    indexer_client.create_or_update_indexer(indexer)
    if status is not None:
        if RESET_INDEXER:
            indexer_client.reset_indexer(INDEXER)
        indexer_client.run_indexer(INDEXER)
    print(f"Started {INDEXER} with {CHUNKER}. Run the status cell next.")

### Wait for completion and inspect the result

The SDK starts indexers asynchronously, so this short loop polls for up to 20 minutes. It ignores the previous
run's result and raises on failure or timeout. This cell never starts or resets an indexer.

In [ ]:
if LIVE:
    deadline = time.monotonic() + 1200
    while time.monotonic() < deadline:
        last = indexer_client.get_indexer_status(INDEXER).last_result
        if last and last.status != "inProgress" and last.end_time and last.end_time != previous_end:
            break
        print("Indexer:", last.status if last else "starting", end="\r")
        time.sleep(15)
    else:
        raise TimeoutError("Indexer not finished after 20 minutes; it may still be running. Rerun this status cell.")
    print(f"\n{CHUNKER}: {last.status}, files {last.item_count}, failed {last.failed_item_count}")
    for warning in last.warnings:
        print("Warning:", warning.message)
    for error in last.errors:
        print("Error:", error.error_message)
    if last.status != "success" or last.failed_item_count:
        raise RuntimeError(f"Indexer failed: {last.error_message or last.status}. Review the errors above.")
    chunks = search_client.get_document_count()
    print("Chunks in index:", chunks)
    if chunks == 0:
        raise RuntimeError("No searchable chunks yet. Rerun this status cell after search refresh. If still empty, "
                           "review the warnings; for the Text Split alternative set CHUNKER='builtIn' and "
                           "RESET_INDEXER=True, then rerun configuration, run and status.")

Add document-version information to the searchable text, so the agent can distinguish current guidance from outdated guidance.
The indexing pipeline splits each PDF into searchable text chunks and creates their embeddings.
This additional cell matches each chunk’s filename to the catalogue.

In [ ]:
if LIVE:
    reg = register.set_index("file_name")
    cols = ["document_id", "title", "document_type", "venue_id", "version", "document_status", "sensitivity"]
    rows = list(search_client.search(search_text="*", select=["chunk_id", "file_name"], top=1000))
    updates = [{"chunk_id": r["chunk_id"], **reg.loc[r["file_name"], cols].to_dict()} for r in rows if r["file_name"] in reg.index]
    for i in range(0, len(updates), 500):
        search_client.merge_documents(documents=updates[i:i + 500])
    print(f"Register fields merged into {len(updates)} of {len(rows)} chunks")

Chunk inspection: chunks per document type and status, then the chunks that hold finding F-0034-1 and the two
versions of the Ridge Ski Jumping Hill wind limit.

In [ ]:
if LIVE:
    time.sleep(2)
    facets = search_client.search(search_text="*", facets=["document_type", "document_status"], top=0).get_facets()
    for name, values in facets.items():
        print(name, {v["value"]: v["count"] for v in values})

    for query, flt in [("F-0034-1 compressor 2 bearing", "document_type eq 'Inspection report'"),
                       ("wind gust limit consecutive minutes", "venue_id eq 'V07'")]:
        print(query)
        hits = search_client.search(search_text=query, filter=flt, top=3,
                                    select=["document_id", "version", "document_status", "page_from", "chunk"])
        display(pd.DataFrame([{"document": h.get("document_id"), "version": h.get("version"),
                               "status": h.get("document_status"), "page": h.get("page_from"),
                               "chunk": (h.get("chunk") or "")[:300]} for h in hits]))

## 4 · Knowledge sources side by side (preview)  *(deck section 3)*

Foundry IQ is the knowledge layer of Azure AI Search: **knowledge sources** describe where content lives, and a
**knowledge base** plans the queries, retrieves from its sources in parallel and returns grounding data or a
synthesised answer with citations. This step registers three sources over the same scenario and puts them in
one knowledge base:

| Knowledge source | Content | Who builds the pipeline |
|---|---|---|
| `games-docs-ks` (search index) | The step 3 index: chunks with register fields | Development team (step 3) |
| `games-onelake-ks` (indexed OneLake) | The same PDF folder | The knowledge source generates data source, skillset, index and indexer |
| `games-ontology-ks` (Fabric ontology) | The ontology from step 1 | Nothing to index: queried live with the user's identity |

The comparison asks the same questions one source at a time (a retrieve request can name the sources to query),
with the same model, reasoning effort and output mode, and reports answer, citations, tokens and time.
The Fabric ontology source needs the user's token on every request (`x-ms-query-source-authorization`) and
doesn't support the *minimal* reasoning effort.

Sources: https://learn.microsoft.com/en-us/azure/search/agentic-knowledge-source-how-to-search-index ·
https://learn.microsoft.com/en-us/azure/search/agentic-knowledge-source-how-to-onelake ·
https://learn.microsoft.com/en-us/azure/search/agentic-knowledge-source-how-to-fabric-ontology ·
https://learn.microsoft.com/en-us/azure/search/agentic-retrieval-how-to-create-knowledge-base ·
https://learn.microsoft.com/en-us/azure/search/agentic-retrieval-how-to-retrieve

In [ ]:
def aoai(deployment, model):
    return {"kind": "azureOpenAI", "azureOpenAIParameters": {"resourceUri": FOUNDRY_OPENAI_URL,
                                                             "deploymentId": deployment, "modelName": model}}

ks_docs = {
    "name": KS_DOCS, "kind": "searchIndex",
    "description": "Venue manuals, inspection reports, transport plans and policies, chunked, with document status.",
    "searchIndexParameters": {"searchIndexName": INDEX, "semanticConfigurationName": "games-semantic",
                              "sourceDataFields": [{"name": f} for f in ("document_id", "title", "document_type", "venue_id",
                                                                         "version", "document_status", "page_from")]},
}
ks_onelake = {
    "name": KS_ONELAKE, "kind": "indexedOneLake",
    "description": "The same PDF folder, indexed by a generated pipeline.",
    "indexedOneLakeParameters": {"fabricWorkspaceId": FABRIC_WORKSPACE_ID, "lakehouseId": FABRIC_LAKEHOUSE_ID,
                                 "targetPath": FABRIC_DOCS_FOLDER,
                                 "ingestionParameters": {"embeddingModel": aoai(FOUNDRY_EMBEDDING_NAME, EMBED_MODEL),
                                                         "chatCompletionModel": aoai("gpt-5.4-mini","gpt-5.4-mini"),
                                                         "contentExtractionMode": "minimal"}},
}
ks_ontology = {
    "name": KS_ONTOLOGY, "kind": "fabricOntology",
    "description": "Games operations ontology: venues, sessions, athletes, assets, findings, rules and live readings.",
    "fabricOntologyParameters": {"workspaceId": FABRIC_WORKSPACE_ID, "ontologyId": FABRIC_ONTOLOGY_ID},
}
if LIVE:
    index_client.create_or_update_knowledge_source(ks_docs)
    index_client.create_or_update_knowledge_source(ks_ontology)   # preview kind: needs the 2026-08-01-preview API version
    # CHARGEABLE: the indexed OneLake source runs its own extraction and embeddings over the 51 PDFs
    index_client.create_or_update_knowledge_source(ks_onelake)
    print("Knowledge sources created:", KS_DOCS, KS_ONTOLOGY, KS_ONELAKE)

The indexed OneLake source builds its pipeline from a fixed template and names the objects after itself
(`games-onelake-ks-datasource`, `-skillset`, `-index`, `-indexer`). The next cell waits for the first
synchronisation and lists the generated index fields, to compare them with the step 3 index.

In [ ]:
if LIVE:
    t0 = time.time()
    while time.time() - t0 < 1800:
        st = index_client.get_knowledge_source_status(KS_ONELAKE).as_dict()   # verify https://learn.microsoft.com/en-us/azure/search/agentic-knowledge-source-how-to-onelake
        last = st.get("lastSynchronizationState") or st.get("last_synchronization_state")
        print(f"  {int(time.time() - t0)} s · {st.get('synchronizationStatus') or st.get('synchronization_status')}", end="\r")
        if last:
            print("\nLast synchronisation:", json.dumps(last, default=str)[:400])
            break
        time.sleep(20)
    generated = index_client.get_index(f"{KS_ONELAKE}-index")
    mine = {f.name for f in index_definition.fields}
    theirs = {f.name for f in generated.fields}
    print("Generated index fields:", ", ".join(sorted(theirs)))
    print("Only in the step 3 index:", ", ".join(sorted(mine - theirs)))

One knowledge base over the three sources. The retrieval instructions tell the planner which source answers
which kind of question; the answer instructions ask for citations and for superseded documents to be ignored.

In [ ]:
kb_definition = {
    "name": KB,
    "description": "Games operations knowledge: ontology plus documents.",
    "knowledgeSources": [{"name": KS_DOCS}, {"name": KS_ONELAKE}, {"name": KS_ONTOLOGY}],
    "models": [aoai("gpt-5.4-mini", "gpt-5.4-mini")],
    "outputMode": "answerSynthesis",
    "retrievalReasoningEffort": {"kind": "low"},
    "retrievalInstructions": ("Use games-ontology-ks for counts, schedules, athletes, assets, findings, business rules and "
                              "live readings. Use the document sources for procedures, operating limits and inspector wording."),
    "answerInstructions": ("Answer in at most five sentences and cite every source. Only documents with status Current are "
                           "authoritative; if a source is Superseded, say so and don't use its values."),
}
if LIVE:
    index_client.create_or_update_knowledge_base(kb_definition)
    print("Knowledge base", KB, "ready")

**Define retrieval for the comparison cells.** This cell sets up the SDK client and function; it does not run
a query or display results. The following cells call `retrieve` with a question and selected sources, then display
the answer, cited documents, reported tokens and elapsed time. The ontology source uses your own sign-in, scoped
to Azure AI Search and passed through the SDK's `headers` argument as `x-ms-query-source-authorization`.
Retrieval errors stop execution with a traceback rather than appearing as answers in the comparison table.

In [ ]:
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient

SOURCE_KIND = {KS_DOCS: "searchIndex", KS_ONELAKE: "indexedOneLake", KS_ONTOLOGY: "fabricOntology"}
CURRENT_ONLY = {KS_DOCS: "document_status eq 'Current'"}

def pick(d, name, default=None):
    snake = re.sub(r"(?<!^)(?=[A-Z])", "_", name).lower()
    return d.get(name, d.get(snake, default)) if isinstance(d, dict) else default

def ref_label(ref):
    data = pick(ref, "sourceData") or {}
    if pick(ref, "type") == "fabricOntology":
        return "ontology"
    return (pick(data, "document_id") or pick(data, "title") or pick(ref, "docKey") or
            (pick(ref, "docUrl") or "").rsplit("/", 1)[-1] or "?")

def retrieve(question, sources, effort="low", output="answerSynthesis", filters=None):
    params = [{"knowledgeSourceName": s, "kind": SOURCE_KIND[s], "includeReferences": True,
               "includeReferenceSourceData": True, **({"filterAddOn": filters[s]} if filters and s in filters else {})}
              for s in sources]
    body = {"messages": [{"role": "user", "content": [{"type": "text", "text": question}]}],
            "knowledgeSourceParams": params, "retrievalReasoningEffort": {"kind": effort},
            "outputMode": output, "includeActivity": True}
    user_token = credential.get_token("https://search.azure.com/.default").token
    t0 = time.perf_counter()
    result = kb_client.retrieve(
        retrieval_request=body,
        headers={"x-ms-query-source-authorization": user_token},
    ).as_dict()
    seconds = round(time.perf_counter() - t0, 1)
    answer = " ".join(pick(c, "text", "") for m in pick(result, "response", []) for c in pick(m, "content", []))
    tokens = sum((pick(a, "inputTokens") or 0) + (pick(a, "outputTokens") or 0) + (pick(a, "reasoningTokens") or 0)
                 for a in pick(result, "activity", []))
    cited = sorted({ref_label(r) for r in pick(result, "references", [])})
    return {"answer": answer[:400], "cited": ", ".join(cited)[:160], "tokens": tokens, "seconds": seconds}

if LIVE:
    kb_client = KnowledgeBaseRetrievalClient(endpoint=AZURE_SEARCH_ENDPOINT, knowledge_base_name=KB,
                                             credential=credential, api_version=SEARCH_API_VERSION)

**Documents: custom index, custom index filtered to Current documents, generated OneLake index.** Q09 and Q13 ask
for wind limits that differ between the Current and the Superseded manual; Q10 asks for a procedure; Q17 asks for
an inspector's wording.

In [ ]:
pd.set_option("display.max_colwidth", 400)
DOC_RUNS = [("custom index", [KS_DOCS], None), ("custom index, Current only", [KS_DOCS], CURRENT_ONLY),
            ("generated OneLake index", [KS_ONELAKE], None)]
if LIVE:
    # CHARGEABLE: 12 knowledge base requests (query planning and answer synthesis)
    rows = []
    for qid in ("Q09", "Q13", "Q10", "Q17"):
        for label, sources, flt in DOC_RUNS:
            rows.append({"question": qid, "source": label, **retrieve(questions.loc[qid].question, sources, filters=flt)})
    doc_results = pd.DataFrame(rows)
    display(doc_results)
    display(questions.loc[["Q09", "Q13", "Q10", "Q17"], ["question", "expected_answer"]])

**Ontology source.** Q05 joins sessions and venue conditions, Q12 needs the `ServiceOverdue` rule, and Q11 needs
the findings extracted by AI functions and the `SessionAtRisk` rule.

In [ ]:
if LIVE:
    # CHARGEABLE: 3 knowledge base requests
    rows = [{"question": qid, **retrieve(questions.loc[qid].question, [KS_ONTOLOGY])} for qid in ("Q05", "Q12", "Q11")]
    display(pd.DataFrame(rows))
    display(questions.loc[["Q05", "Q12", "Q11"], ["question", "expected_answer"]])

**Reasoning effort.** Q11 needs both the ontology (which sessions) and the inspection report (what the inspector
recommended). The same question runs with the custom index (Current only) plus the ontology at *low* and
*medium* effort; *minimal* runs on the documents alone, because it needs extractive output and the ontology source
doesn't support it. Higher effort plans more sub-queries: compare citations, tokens and time.

In [ ]:
if LIVE:
    # CHARGEABLE: 3 knowledge base requests
    q11 = questions.loc["Q11"].question
    rows = [{"effort": e, **retrieve(q11, [KS_DOCS, KS_ONTOLOGY], effort=e, filters=CURRENT_ONLY)} for e in ("low", "medium")]
    rows.append({"effort": "minimal (documents only, extractive)",
                 **retrieve(q11, [KS_DOCS], effort="minimal", output="extractiveData", filters=CURRENT_ONLY)})
    display(pd.DataFrame(rows))
    print("Expected:", questions.loc["Q11"].expected_answer)

**What to look for**
- The filter on `document_status` is what keeps the superseded wind limits out of Q09 and Q13. The generated
  index has no register fields, so it can't apply that filter.
- The ontology source answers with data and returns the raw rows behind the answer; the document sources answer
  with cited chunks.
- From step 5 onward, the agents use the custom index (Current only) and the ontology. Using both document sources
  at once would return duplicate chunks of the same files.

## Cleanup

Deletes the knowledge base, the three knowledge sources, the objects generated by the indexed OneLake source and
the step 3 pipeline (indexer, skillset, data source, index). Set `CLEANUP = True` to run it; later steps of this
notebook reuse these objects, so keep them until the end of the demo.

**Not deleted:** the Azure AI Search service (billed per hour for Basic and higher tiers), the Foundry model
deployments, the Fabric workspace and its items (lakehouse, eventhouse, semantic model, ontology) and the Fabric
capacity, which bills while it runs.

In [ ]:
# DESTRUCTIVE: deletes the search objects created in steps 3 and 4
CLEANUP = False
if LIVE and CLEANUP:
    for action, name in [(index_client.delete_knowledge_base, KB),
                         (index_client.delete_knowledge_source, KS_DOCS),
                         (index_client.delete_knowledge_source, KS_ONELAKE),
                         (index_client.delete_knowledge_source, KS_ONTOLOGY),
                         (indexer_client.delete_indexer, f"{KS_ONELAKE}-indexer"),
                         (indexer_client.delete_skillset, f"{KS_ONELAKE}-skillset"),
                         (indexer_client.delete_data_source_connection, f"{KS_ONELAKE}-datasource"),
                         (index_client.delete_index, f"{KS_ONELAKE}-index"),
                         (indexer_client.delete_indexer, INDEXER), (indexer_client.delete_skillset, SKILLSET),
                         (indexer_client.delete_data_source_connection, DATASOURCE), (index_client.delete_index, INDEX)]:
        try:
            action(name)
            print("deleted", name)
        except Exception as e:
            print("skipped", name, "-", type(e).__name__)

## What you built

| Step | Built / shown | Azure resource touched |
|---|---|---|
| 0 | Settings, clients, sample register and questions | Foundry project (read) |
| 1 | Ontology MCP tool list and one question | Fabric ontology (read) |
| 2 | Fabric IQ MCP tool list and one discovery call | Fabric semantic models (read) |
| 3 | `games-docs` index, skillset, OneLake data source and indexer; register fields merged into chunks | Azure AI Search; Foundry resource (extraction, embeddings) |
| 4 | Three knowledge sources, the `games-kb` knowledge base and three comparisons | Azure AI Search; Foundry resource (chat model, embeddings); Fabric ontology (read) |

## Sources and tested versions

Written against `azure-search-documents` 12.1.0b2 (REST API version 2026-08-01-preview), `mcp` 1.x
(`streamablehttp_client`, as in notebooks 1 and 8) and `azure-ai-projects` 2.x. Not yet executed against Azure.

- https://learn.microsoft.com/en-us/fabric/iq/ontology/how-to-use-ontology-mcp-server
- https://learn.microsoft.com/en-us/fabric/iq/connectors/fabric-iq-mcp
- https://learn.microsoft.com/en-us/azure/search/search-how-to-index-onelake-files
- https://learn.microsoft.com/en-us/azure/search/cognitive-search-skill-content-understanding
- https://learn.microsoft.com/en-us/azure/search/search-how-to-semantic-chunking-content-understanding
- https://learn.microsoft.com/en-us/azure/search/cognitive-search-skill-azure-openai-embedding
- https://learn.microsoft.com/en-us/azure/search/search-how-to-define-index-projections
- https://learn.microsoft.com/en-us/azure/search/cognitive-search-attach-cognitive-services
- https://learn.microsoft.com/en-us/azure/search/agentic-retrieval-how-to-create-index
- https://learn.microsoft.com/en-us/azure/search/agentic-knowledge-source-how-to-search-index
- https://learn.microsoft.com/en-us/azure/search/agentic-knowledge-source-how-to-onelake
- https://learn.microsoft.com/en-us/azure/search/agentic-knowledge-source-how-to-fabric-ontology
- https://learn.microsoft.com/en-us/azure/search/agentic-retrieval-how-to-create-knowledge-base
- https://learn.microsoft.com/en-us/azure/search/agentic-retrieval-how-to-retrieve
- https://github.com/Azure/azure-sdk-for-python/blob/main/sdk/search/azure-search-documents/CHANGELOG.md

## Before you build on this: re-check …
- **Fabric sign-in scopes** for the ontology MCP endpoint and Fabric IQ MCP (`# verify` in step 1): the
  documentation names only the delegated Power BI permissions for Fabric IQ MCP and shows interactive sign-in for
  the ontology endpoint.
- **Content Understanding with OneLake:** whether the OneLake indexer passes the file bytes to the skill; step 3
  offers `CHUNKER = "builtIn"` as an explicit alternative when it doesn't.
- **Preview surfaces:** the Fabric ontology knowledge source, answer synthesis and reasoning effort need REST API
  version 2026-08-01-preview and the pre-release client; class names and the `fabricOntology` parameter kind can
  change between betas.
- **`mcp` 2.0** removed `streamablehttp_client`; the kit caps `mcp` below 2 until notebooks 1, 8 and 9 move to
  `streamable_http_client`.